# Seedance 2.0 Reference-to-Video API 使用指南

本 Notebook 演示如何通过 **七牛 AIToken** 平台调用 **Seedance 2.0 (Reference-to-Video)** 模型 API。

## 模型简介

Seedance 2.0 Reference-to-Video 是字节跳动推出的多模态参考视频生成模型，支持以下能力：

- **图片参考生成视频**：使用参考图片指导视频生成，通过 `@Image1` 等在 prompt 中引用
- **视频参考生成视频**：使用参考视频指导视频生成，通过 `@Video1` 等在 prompt 中引用
- **音频参考生成视频**：使用参考音频指导视频配音，通过 `@Audio1` 等在 prompt 中引用
- **多模态混合参考**：支持同时传入图片、视频、音频作为参考素材
- **同步音频生成**：可自动生成与视频同步的音频

**API 端点**：`bytedance/seedance-2.0/reference-to-video`

## 1. 环境准备

首先安装 `fal-client` 依赖包，并设置 API Key 和七牛 AIToken 队列服务地址。

In [ ]:
# 安装 fal-client
%pip install fal-client -q

In [ ]:
import os

# 设置七牛 AIToken 队列服务地址
os.environ["FAL_QUEUE_RUN_HOST"] = "api.qnaigc.com/queue"

# 设置 FAL_KEY 环境变量
# 你也可以在终端中通过 export FAL_KEY="xxx" 来设置
os.environ["FAL_KEY"] = os.environ.get("QINIU_API_KEY") or (_ for _ in ()).throw(ValueError("环境变量 QINIU_API_KEY 未设置"))

In [ ]:
import fal_client
from IPython.display import Video, display

In [ ]:
import time

def run_fal_task(endpoint, arguments, poll_interval=5):
    """提交任务并轮询等待结果返回"""
    # 步骤 1：提交请求
    handler = fal_client.submit(endpoint, arguments=arguments)
    request_id = handler.request_id
    print(f"请求已提交，request_id: {request_id}")

    # 步骤 2：轮询任务状态
    while True:
        status = fal_client.status(endpoint, request_id, with_logs=True)
        print(f"当前状态: {type(status).__name__}")

        if isinstance(status, fal_client.Completed):
            print("任务完成!")
            break

        if hasattr(status, "logs") and status.logs:
            for log in status.logs:
                print(log["message"])

        time.sleep(poll_interval)

    # 步骤 3：获取结果
    return fal_client.result(endpoint, request_id)

## 2. 图片参考生成视频

通过 `image_urls` 传入参考图片，在 prompt 中使用 `@Image1`、`@Image2` 等引用对应图片。

- 最多支持 9 张参考图片
- 每张图片不超过 30MB
- 在 prompt 中通过 `@Image1`、`@Image2` 等引用

In [ ]:
# 图片参考生成视频
result_image_ref = run_fal_task(
    "bytedance/seedance-2.0/reference-to-video",
    arguments={
        "prompt": "@Image1 中的角色在花园里快乐地奔跑，阳光透过树叶洒落，温馨的画面",
        "image_urls": [
            "https://v3.fal.media/files/zebra/sgsdKvPigPhJ1S7Hl5bWc_first_frame_q1.png",
        ],
    },
)

# 打印返回结果
print(result_image_ref)

In [ ]:
# 展示生成的视频
video_url = result_image_ref["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

### 参数说明

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `prompt` | string | *必填* | 提示词，通过 `@Image1`、`@Video1`、`@Audio1` 引用参考素材 |
| `image_urls` | list\<string\> | 可选 | 参考图片 URL 列表（最多 9 张，每张 ≤30MB） |
| `video_urls` | list\<string\> | 可选 | 参考视频 URL 列表（最多 3 个，总时长 2-15 秒） |
| `audio_urls` | list\<string\> | 可选 | 参考音频 URL 列表（最多 3 个，总时长 ≤15 秒） |
| `resolution` | enum | `720p` | 分辨率：`480p`、`720p` |
| `duration` | enum | `auto` | 视频时长：`auto` 或 `4`-`15` 秒 |
| `aspect_ratio` | enum | `auto` | 宽高比：`auto`、`21:9`、`16:9`、`4:3`、`1:1`、`3:4`、`9:16` |
| `generate_audio` | boolean | `true` | 是否生成与视频同步的音频 |
| `seed` | integer | 随机 | 随机种子，用于结果复现 |

## 3. 多图片参考生成视频

传入多张参考图片，在 prompt 中分别引用，让模型综合多个图片的内容生成视频。

In [ ]:
# 多图片参考生成视频
result_multi_image = run_fal_task(
    "bytedance/seedance-2.0/reference-to-video",
    arguments={
        "prompt": "@Image1 中的角色走向 @Image2 中的场景，镜头缓缓推进，电影质感",
        "image_urls": [
            "https://v3.fal.media/files/zebra/sgsdKvPigPhJ1S7Hl5bWc_first_frame_q1.png",
            "https://v3.fal.media/files/kangaroo/CASBu_OmOnZ8IafirarFL_last_frame_q1.png",
        ],
        "aspect_ratio": "16:9",
        "duration": "8",
    },
)

# 展示结果
video_url = result_multi_image["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 4. 视频参考生成视频

通过 `video_urls` 传入参考视频，在 prompt 中使用 `@Video1` 等引用。模型会参考视频的风格、运动方式等生成新视频。

- 最多支持 3 个参考视频
- 参考视频总时长需在 2-15 秒之间

In [ ]:
# 视频参考生成视频
result_video_ref = run_fal_task(
    "bytedance/seedance-2.0/reference-to-video",
    arguments={
        "prompt": "按照 @Video1 的风格和运镜方式，增加一个小狗",
        "video_urls": [
            "https://aitoken-public.qnaigc.com/example/generate-video/the-little-dog-is-running-on-the-lawn.mp4",
        ],
        "aspect_ratio": "16:9",
        "duration": "8",
    },
)

# 展示结果
video_url = result_video_ref["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 5. 多模态混合参考

同时传入图片、视频、音频作为参考素材，在 prompt 中分别引用，让模型综合多种模态的信息生成视频。

In [ ]:
# 多模态混合参考
result_multimodal = run_fal_task(
    "bytedance/seedance-2.0/reference-to-video",
    arguments={
        "prompt": "@Image1 中的角色按照 @Video1 的运动方式跳舞，配合 @Audio1 的节奏",
        "image_urls": [
            "https://v3.fal.media/files/zebra/sgsdKvPigPhJ1S7Hl5bWc_first_frame_q1.png",
        ],
        "video_urls": [
            "https://v3.fal.media/files/elephant/HBxJMDdrX-ijuXsgYgyXL.mp4",
        ],
        "audio_urls": [
            "https://v3.fal.media/files/rabbit/K2Z3MZxj7BjZ7B75jyK2d.wav",
        ],
        "aspect_ratio": "16:9",
        "duration": "10",
        "resolution": "720p",
    },
)

# 展示结果
video_url = result_multimodal["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 6. 高级用法

使用更多可选参数来精细控制视频生成效果。

In [ ]:
# 高级用法：使用更多参数
result_advanced = run_fal_task(
    "bytedance/seedance-2.0/reference-to-video",
    arguments={
        "prompt": "@Image1 中的角色在雪山顶上眺望远方，镜头从背后缓缓拉远，壮丽的雪山全景展现",
        "image_urls": [
            "https://v3.fal.media/files/zebra/sgsdKvPigPhJ1S7Hl5bWc_first_frame_q1.png",
        ],
        "resolution": "720p",          # 720p 分辨率
        "duration": "12",              # 视频时长 12 秒
        "aspect_ratio": "21:9",        # 超宽屏比例
        "generate_audio": True,         # 生成同步音频
        "seed": 42,                     # 固定随机种子，便于复现
    },
)

# 展示结果
video_url = result_advanced["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 7. 队列模式 — 手动分步调用

上面的 `run_fal_task` 封装了完整的队列流程。如果你需要更精细的控制（例如在提交后做其他事情，稍后再取结果），可以手动分步调用。

In [ ]:
# 步骤 1：提交请求
handler = fal_client.submit(
    "bytedance/seedance-2.0/reference-to-video",
    arguments={
        "prompt": "@Image1 中的角色在海边散步，海浪拍打着沙滩，夕阳西下",
        "image_urls": [
            "https://v3.fal.media/files/zebra/sgsdKvPigPhJ1S7Hl5bWc_first_frame_q1.png",
        ],
        "aspect_ratio": "16:9",
    },
)

# 获取请求 ID，用于后续查询
request_id = handler.request_id
print(f"请求已提交，request_id: {request_id}")

In [ ]:
# 步骤 2：查询任务状态
import time

while True:
    status = fal_client.status(
        "bytedance/seedance-2.0/reference-to-video",
        request_id,
        with_logs=True,
    )
    print(f"当前状态: {type(status).__name__}")

    if isinstance(status, fal_client.Completed):
        print("任务完成!")
        break

    # 每 5 秒查询一次
    time.sleep(5)

In [ ]:
# 步骤 3：获取结果
result_async = fal_client.result(
    "bytedance/seedance-2.0/reference-to-video",
    request_id,
)

# 展示生成的视频
video_url = result_async["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 8. Schema 参考

### Input Schema

```json
{
  "prompt": "string (必填, 通过 @Image1/@Video1/@Audio1 引用参考素材)",
  "image_urls": ["string (参考图片 URL, 最多 9 张, 每张 ≤30MB)"],
  "video_urls": ["string (参考视频 URL, 最多 3 个, 总时长 2-15 秒)"],
  "audio_urls": ["string (参考音频 URL, 最多 3 个, 总时长 ≤15 秒)"],
  "resolution": "480p | 720p (可选, 默认 720p)",
  "duration": "auto | 4-15 (可选, 默认 auto)",
  "aspect_ratio": "auto | 21:9 | 16:9 | 4:3 | 1:1 | 3:4 | 9:16 (可选, 默认 auto)",
  "generate_audio": "boolean (可选, 默认 true)",
  "seed": "integer (可选, 默认随机)"
}
```

### Output Schema

```json
{
  "video": {
    "url": "string (视频文件 URL)",
    "content_type": "string (MIME 类型)",
    "file_name": "string (文件名)",
    "file_size": "integer (文件大小)"
  },
  "seed": "integer (使用的随机种子)"
}
```

### 素材引用方式

| 素材类型 | 引用格式 | 说明 |
|----------|----------|------|
| 图片 | `@Image1`、`@Image2`、... | 按 `image_urls` 列表顺序对应 |
| 视频 | `@Video1`、`@Video2`、... | 按 `video_urls` 列表顺序对应 |
| 音频 | `@Audio1`、`@Audio2`、... | 按 `audio_urls` 列表顺序对应 |

### 更多资源

- [七牛 AIToken 文档](https://developer.qiniu.com/aitokenapi)
- [fal-client Python 文档](https://docs.fal.ai/reference/client-libraries/python/fal_client)
- [Seedance 2.0 模型页面](https://fal.ai/models/bytedance/seedance-2.0)